# DaTSCAN — Fase 6: CNN 2.5D sobre cortes originales de C2

Experimento exploratorio para determinar si el protocolo C2 conserva señal diagnóstica en sus cortes axiales originales que se pierde o distorsiona al construir el volumen 3D interpolado.

La entrada contiene siete cortes axiales originales alrededor del máximo bilateral y siete mapas de diferencia izquierda–derecha (14 canales). Se utiliza CV estratificada interna de cinco folds exclusivamente dentro de C2.

> **Alcance:** C2 ya fue utilizado para decidir este experimento. Por ello, esta CV sirve para comparar representaciones dentro de C2, pero no constituye una nueva validación externa independiente frente a un protocolo desconocido.

## 0. Dependencias

In [ ]:
# Descomente si falta alguna dependencia.
# %pip install nibabel scipy torch numpy pandas scikit-learn matplotlib seaborn

## 1. Librerías, reproducibilidad y configuración

In [ ]:
from pathlib import Path
import os
import copy, gc, random, time, warnings
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter, gaussian_filter1d, label
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

warnings.filterwarnings('ignore',category=FutureWarning)
sns.set_theme(style='whitegrid')
SEED=20260910
def seed_everything(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_everything()
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:',torch.__version__,'| dispositivo:',DEVICE)
if DEVICE.type=='cuda':print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
import sys
REPO_ROOT=Path.cwd().resolve()
if REPO_ROOT.name=='notebooks':REPO_ROOT=REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:sys.path.insert(0,str(REPO_ROOT))
from src.config import DATA_ROOT
PROJECT_DIR=DATA_ROOT/'latent_protocol_cv'
FOLDS_CSV=PROJECT_DIR/'outputs'/'train_protocol_folds.csv'
METADATA_CSV=PROJECT_DIR/'outputs'/'protocol_metadata_clustered.csv'
OUTPUT_DIR=PROJECT_DIR/'cnn25d_original_C2_v1'
CACHE_DIR=OUTPUT_DIR/'cache_14ch_80x80'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True);CACHE_DIR.mkdir(parents=True,exist_ok=True)

TARGET_CLUSTER=2
N_SLICES=7
SLICE_OFFSETS=np.arange(-(N_SLICES//2),N_SLICES//2+1)
CROP_SIZE=80
BATCH_SIZE=16
NUM_WORKERS=0
MAX_EPOCHS=50
PATIENCE=8
LEARNING_RATE=5e-4
WEIGHT_DECAY=1e-4
INNER_VALID_FRACTION=.15
N_SPLITS=5

for p in (FOLDS_CSV,METADATA_CSV):
    print(p,'| existe:',p.exists())
    if not p.exists():raise FileNotFoundError(p)
print('Offsets axiales originales:',SLICE_OFFSETS,'| salida:',OUTPUT_DIR)

## 2. Cohorte C2 y rutas originales

In [ ]:
folds=pd.read_csv(FOLDS_CSV)
metadata=pd.read_csv(METADATA_CSV)
manifest=(folds.merge(metadata[['uid','nifti_path','shape_x','shape_y','shape_z','spacing_x','spacing_y','spacing_z']],
                      on='uid',how='left',validate='one_to_one')
          .query('protocol_cluster == @TARGET_CLUSTER').reset_index(drop=True))
if len(manifest)!=256:print('ADVERTENCIA: se esperaban 256 casos C2 y se encontraron',len(manifest))
if manifest.uid.duplicated().any():raise ValueError('UID duplicados.')
missing=manifest[~manifest.nifti_path.map(lambda p:Path(p).exists())]
if len(missing):display(missing.head());raise FileNotFoundError(f'Faltan {len(missing)} NIfTI.')
print('C2:',len(manifest),'| normales:',int((manifest.target==0).sum()),'| patológicos:',int((manifest.target==1).sum()))
display(manifest.groupby('target').agg(n=('uid','size')))
display(manifest[['shape_x','shape_y','shape_z','spacing_x','spacing_y','spacing_z']].describe().T)

## 3. Lectura, normalización y localización bilateral en el NIfTI original

In [ ]:
def largest_component(mask):
    components,n=label(mask)
    if n==0:return mask
    sizes=np.bincount(components.ravel());sizes[0]=0
    return components==sizes.argmax()

def load_original(path):
    image=nib.as_closest_canonical(nib.load(str(path)))
    volume=np.asarray(image.dataobj,dtype=np.float32)
    volume=np.nan_to_num(volume,nan=0,posinf=0,neginf=0)
    volume=np.maximum(volume,0)
    positive=volume[volume>0]
    if positive.size<100:raise ValueError('Volumen prácticamente vacío.')
    scale=float(np.percentile(positive,99.5))
    volume=np.clip(volume/max(scale,1e-6),0,1)
    return volume,tuple(float(v) for v in image.header.get_zooms()[:3])

def head_center_xy(volume):
    mip=volume.max(axis=2)
    positive=mip[mip>0]
    threshold=max(.02,float(np.percentile(positive,20)))
    mask=largest_component(mip>threshold)
    coords=np.argwhere(mask)
    if len(coords)<100:return np.asarray(volume.shape[:2],dtype=float)/2
    return np.median(coords,axis=0)

def crop_xy(slice2d,center,size=CROP_SIZE):
    out=np.zeros((size,size),dtype=np.float32)
    center=np.rint(center).astype(int);start=center-size//2
    src=[];dst=[]
    for st,n in zip(start,slice2d.shape):
        a,b=max(0,st),min(n,st+size);da=max(0,-st);db=da+max(0,b-a)
        src.append(slice(a,b));dst.append(slice(da,db))
    out[tuple(dst)]=slice2d[tuple(src)]
    return out

def locate_original_striatum(volume):
    center=head_center_xy(volume)
    # Trabajar en recorte nativo común evita que el localizador dependa del tamaño de matriz.
    cropped=np.stack([crop_xy(volume[:,:,z],center) for z in range(volume.shape[2])],axis=2)
    smooth=gaussian_filter(cropped,sigma=(1.4,1.4,.55))
    n=CROP_SIZE;middle=n//2
    z0=max(N_SLICES//2,int(round(.18*volume.shape[2])))
    z1=min(volume.shape[2]-N_SLICES//2,int(round(.88*volume.shape[2])))
    y0,y1=int(.25*n),int(.78*n)
    half_distances=range(7,13)
    candidates=[]
    for z in range(z0,z1):
        slab=smooth[:,:,max(0,z-1):min(smooth.shape[2],z+2)].mean(axis=2)
        base_l=float(np.percentile(slab[12:middle,y0:y1],55))
        base_r=float(np.percentile(slab[middle:n-12,y0:y1],55))
        best=None
        for d in half_distances:
            lx,rx=middle-d,middle+d
            for y in range(y0+2,y1-2):
                cl=max(float(slab[lx,y]-base_l),0);cr=max(float(slab[rx,y]-base_r),0)
                signal=max(cl,cr)+.45*min(cl,cr)
                balance=np.sqrt((min(cl,cr)+1e-4)/(max(cl,cr)+1e-4))
                ypos=(y-(y0+2))/max(1,(y1-3)-(y0+2));yp=max(np.sin(np.pi*ypos),.10)**.30
                zpos=(z-z0)/max(1,z1-z0-1);zp=max(np.sin(np.pi*zpos),.08)**.35
                score=signal*(.65+.35*balance)*yp*zp
                if best is None or score>best['score']:
                    best={'score':score,'z':z,'y':y,'lx':lx,'rx':rx,'cl':cl,'cr':cr,'d':d}
        candidates.append(best)
    profile=gaussian_filter1d(np.asarray([c['score'] for c in candidates]),sigma=.8)
    idx=int(np.argmax(profile));best=candidates[idx];best['z']=z0+idx
    best['profile']=profile;best['z_range']=(z0,z1);best['center_original']=center
    best['prominence']=float(profile[idx]/(np.median(profile)+1e-6))
    best['cropped_volume']=cropped
    return best

def build_25d(path):
    volume,spacing=load_original(path);loc=locate_original_striatum(volume)
    cuts=[]
    for off in SLICE_OFFSETS:
        z=int(np.clip(loc['z']+off,0,volume.shape[2]-1))
        cuts.append(loc['cropped_volume'][:,:,z].T)
    intensity=np.stack(cuts).astype(np.float32)
    # La dimensión final es izquierda-derecha después de transponer.
    asym=np.abs(intensity-intensity[:,:,::-1])
    x=np.concatenate([intensity,asym],axis=0)
    info={k:loc[k] for k in ('z','y','lx','rx','cl','cr','d','prominence')}
    info.update(original_shape=str(tuple(volume.shape)),spacing=str(spacing),
                center_x=float(loc['center_original'][0]),center_y=float(loc['center_original'][1]))
    return x,info,loc

## 4. Auditoría visual obligatoria antes de extraer todos los casos

In [ ]:
sample=manifest.groupby('target',group_keys=False).sample(n=4,random_state=SEED)
fig,axes=plt.subplots(len(sample),3,figsize=(11,3*len(sample)))
audit=[]
for i,(_,row) in enumerate(sample.reset_index(drop=True).iterrows()):
    x,info,loc=build_25d(row.nifti_path);z=loc['z'];mid=N_SLICES//2
    axes[i,0].imshow(x[mid],cmap='magma',origin='lower',vmin=0,vmax=1)
    axes[i,0].add_patch(Circle((loc['lx'],loc['y']),5,fill=False,color='cyan'))
    axes[i,0].add_patch(Circle((loc['rx'],loc['y']),5,fill=False,color='lime'))
    axes[i,0].set_title(f"{row.uid} | y={row.target} | z={z}")
    axes[i,1].imshow(x[N_SLICES+mid],cmap='viridis',origin='lower')
    axes[i,1].set_title('|I − espejo|')
    zr=range(*loc['z_range']);axes[i,2].plot(zr,loc['profile']);axes[i,2].axvline(z,color='crimson',ls='--')
    axes[i,2].set(title=f"Prominencia={loc['prominence']:.2f}",xlabel='corte axial original')
    axes[i,0].axis('off');axes[i,1].axis('off');audit.append({'uid':row.uid,'target':row.target,**info})
plt.tight_layout();plt.savefig(OUTPUT_DIR/'auditoria_localizacion_original_C2.png',dpi=170,bbox_inches='tight');plt.show()
audit_df=pd.DataFrame(audit);display(audit_df);audit_df.to_csv(OUTPUT_DIR/'auditoria_localizacion_original_C2.csv',index=False)

In [ ]:
# Revise que ambos estriados estén dentro del recorte central y que el corte seleccionado sea razonable.
# Después cambie a True y ejecute desde esta celda.
VISUAL_AUDIT_APPROVED=False
if not VISUAL_AUDIT_APPROVED:
    raise RuntimeError('Revise la figura anterior y cambie VISUAL_AUDIT_APPROVED=True.')

## 5. Extracción y caché reproducible de las entradas 2.5D

In [ ]:
records=[];errors=[];start=time.time()
for i,row in manifest.iterrows():
    output=CACHE_DIR/f'{row.uid}.npz';reused=False
    try:
        if output.exists():
            with np.load(output) as saved:x=saved['x']
            reused=x.shape==(2*N_SLICES,CROP_SIZE,CROP_SIZE) and np.isfinite(x).all()
        if not reused:
            x,info,_=build_25d(row.nifti_path)
            np.savez_compressed(output,x=x.astype(np.float16))
        else:info={}
        records.append({'uid':row.uid,'target':int(row.target),'cache_path':str(output.resolve()),'reused':reused,**info})
    except Exception as exc:
        errors.append({'uid':row.uid,'error_type':type(exc).__name__,'error':str(exc)})
    if (i+1)%25==0 or i+1==len(manifest):
        print(f'{i+1}/{len(manifest)} | correctos={len(records)} | errores={len(errors)} | {(time.time()-start)/60:.1f} min')
        pd.DataFrame(records).to_csv(OUTPUT_DIR/'extraction_checkpoint.csv',index=False)
extraction=pd.DataFrame(records);error_df=pd.DataFrame(errors)
extraction.to_csv(OUTPUT_DIR/'extraction_manifest.csv',index=False);error_df.to_csv(OUTPUT_DIR/'extraction_errors.csv',index=False)
if len(errors):display(error_df);raise RuntimeError(f'Fallaron {len(errors)} estudios.')
manifest=manifest.merge(extraction[['uid','cache_path']],on='uid',how='left',validate='one_to_one')
print('Extracción completa:',len(extraction))

## 6. Dataset 2.5D y arquitectura compacta

In [ ]:
class DaTSCAN25D(Dataset):
    def __init__(self,frame,augment=False):self.frame=frame.reset_index(drop=True);self.augment=augment
    def __len__(self):return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with np.load(row.cache_path) as saved:x=torch.from_numpy(saved['x'].astype(np.float32))
        if self.augment:
            if torch.rand(())<.5:x=torch.flip(x,dims=(2,))
            scale=float(torch.empty(1).uniform_(.96,1.04));x[:N_SLICES]*=scale
            if torch.rand(())<.3:x[:N_SLICES]+=torch.randn_like(x[:N_SLICES])*.005
            x=torch.clamp(x,0,1)
        return x,torch.tensor(float(row.target),dtype=torch.float32),str(row.uid)

def loader(frame,augment,shuffle):
    return DataLoader(DaTSCAN25D(frame,augment),batch_size=BATCH_SIZE,shuffle=shuffle,
                      num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))

class Block2D(nn.Module):
    def __init__(self,cin,cout):
        super().__init__();groups=max(g for g in (8,4,2,1) if cout%g==0)
        self.net=nn.Sequential(nn.Conv2d(cin,cout,3,padding=1,bias=False),nn.GroupNorm(groups,cout),nn.SiLU(),
            nn.Conv2d(cout,cout,3,padding=1,bias=False),nn.GroupNorm(groups,cout),nn.SiLU(),nn.MaxPool2d(2))
    def forward(self,x):return self.net(x)

class SmallCNN25D(nn.Module):
    def __init__(self):
        super().__init__();self.features=nn.Sequential(Block2D(2*N_SLICES,24),Block2D(24,48),Block2D(48,96))
        self.head=nn.Sequential(nn.Linear(192,64),nn.SiLU(),nn.Dropout(.30),nn.Linear(64,1))
    def forward(self,x):
        x=self.features(x);x=torch.cat([F.adaptive_avg_pool2d(x,1),F.adaptive_max_pool2d(x,1)],1).flatten(1)
        return self.head(x).squeeze(1)

model=SmallCNN25D().to(DEVICE)
print('Parámetros:',sum(p.numel() for p in model.parameters() if p.requires_grad))
x,y,_=next(iter(loader(manifest.head(BATCH_SIZE),False,False)))
with torch.no_grad():out=model(x.to(DEVICE))
print('Entrada:',tuple(x.shape),'| salida:',tuple(out.shape))
del model,x,y,out
if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 7. Prueba obligatoria de sobreajuste en 16 casos

In [ ]:
tiny=manifest.groupby('target',group_keys=False).sample(n=8,random_state=SEED)
tiny_loader=loader(tiny,False,True);tiny_model=SmallCNN25D().to(DEVICE)
tiny_opt=torch.optim.AdamW(tiny_model.parameters(),lr=1e-3,weight_decay=0)
for epoch in range(1,61):
    tiny_model.train()
    for x,y,_ in tiny_loader:
        x=x.to(DEVICE);y=y.to(DEVICE);tiny_opt.zero_grad();loss=F.binary_cross_entropy_with_logits(tiny_model(x),y)
        loss.backward();tiny_opt.step()
    if epoch in (1,10,20,40,60):print(f'Epoch {epoch:03d} | loss={loss.item():.4f}')
if loss.item()>.10:raise RuntimeError('La red no logró memorizar 16 casos; revise el pipeline antes de la CV.')
del tiny_model,tiny_opt,x,y;gc.collect()
if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 8. Entrenamiento con validación interna para early stopping

In [ ]:
@torch.no_grad()
def predict(model,data_loader):
    model.eval();ps=[];ys=[];uids=[]
    for x,y,u in data_loader:
        ps.extend(torch.sigmoid(model(x.to(DEVICE,non_blocking=True))).cpu().numpy());ys.extend(y.numpy());uids.extend(u)
    return np.asarray(ps),np.asarray(ys,dtype=int),uids

def train_outer(train_frame,valid_frame,fold):
    seed_everything(SEED+fold)
    split=StratifiedShuffleSplit(n_splits=1,test_size=INNER_VALID_FRACTION,random_state=SEED+fold)
    a,b=next(split.split(train_frame,train_frame.target));inner_train=train_frame.iloc[a];inner_valid=train_frame.iloc[b]
    train_loader=loader(inner_train,True,True);inner_loader=loader(inner_valid,False,False);valid_loader=loader(valid_frame,False,False)
    model=SmallCNN25D().to(DEVICE);opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=2,min_lr=1e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
    best=np.inf;best_state=None;best_epoch=0;wait=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();total=0;n=0
        for x,y,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True);opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type=='cuda'):
                logits=model(x);loss=F.binary_cross_entropy_with_logits(logits,y)
            scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),5)
            scaler.step(opt);scaler.update();total+=loss.item()*len(y);n+=len(y)
        p,yv,_=predict(model,inner_loader);p=np.clip(p,1e-6,1-1e-6);inner_loss=log_loss(yv,p)
        scheduler.step(inner_loss);history.append({'fold':fold,'epoch':epoch,'train_loss':total/n,'inner_logloss':inner_loss,'inner_auc':roc_auc_score(yv,p),'lr':opt.param_groups[0]['lr']})
        print(f'C2 F{fold} E{epoch:02d} | train={total/n:.4f} | inner={inner_loss:.4f}')
        if inner_loss<best-1e-4:best=inner_loss;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict());wait=0
        else:wait+=1
        if wait>=PATIENCE:break
    model.load_state_dict(best_state);p,yv,uids=predict(model,valid_loader);p=np.clip(p,1e-6,1-1e-6)
    preds=pd.DataFrame({'uid':uids,'target':yv,'prediction':p,'cv_fold_C2':fold})
    metrics={'fold':fold,'n_train':len(train_frame),'n_valid':len(valid_frame),'best_epoch':best_epoch,'inner_best_logloss':best,
             'logloss':log_loss(yv,p),'auc':roc_auc_score(yv,p),'brier':brier_score_loss(yv,p),'prevalence':yv.mean()}
    return model,preds,pd.DataFrame(history),metrics

## 9. CV estratificada interna de C2 con reanudación

In [ ]:
cv=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
oof=np.full(len(manifest),np.nan);uid_index={u:i for i,u in enumerate(manifest.uid)};metric_rows=[]
for fold,(tr,va) in enumerate(cv.split(manifest,manifest.target)):
    pred_path=OUTPUT_DIR/f'C2_fold{fold}_predictions.csv'
    if pred_path.exists():
        preds=pd.read_csv(pred_path);reused=True
        if set(preds.uid)!=set(manifest.iloc[va].uid):raise ValueError(f'Predicciones incompatibles: {pred_path}')
        y=preds.target.astype(int);p=preds.prediction
        metrics={'fold':fold,'n_train':len(tr),'n_valid':len(va),'best_epoch':np.nan,'inner_best_logloss':np.nan,
                 'logloss':log_loss(y,p),'auc':roc_auc_score(y,p),'brier':brier_score_loss(y,p),'prevalence':y.mean()}
    else:
        model,preds,history,metrics=train_outer(manifest.iloc[tr],manifest.iloc[va],fold);reused=False
        torch.save(model.state_dict(),OUTPUT_DIR/f'C2_fold{fold}.pt');preds.to_csv(pred_path,index=False);history.to_csv(OUTPUT_DIR/f'C2_fold{fold}_history.csv',index=False)
        del model;gc.collect()
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    metrics['reused']=reused;metric_rows.append(metrics)
    for u,p in zip(preds.uid,preds.prediction):oof[uid_index[u]]=p
    pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR/'C2_fold_metrics_checkpoint.csv',index=False);display(pd.DataFrame([metrics]))
if np.isnan(oof).any():raise RuntimeError('OOF incompleto.')
oof_frame=pd.DataFrame({'uid':manifest.uid,'target':manifest.target,'protocol_cluster':TARGET_CLUSTER,'prediction_25d':oof})
oof_frame.to_csv(OUTPUT_DIR/'C2_oof_25d.csv',index=False);pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR/'C2_fold_metrics.csv',index=False)
overall=pd.DataFrame([{'model':'CNN25D_original_C2','n':len(oof),'logloss_oof':log_loss(manifest.target,oof),
                       'auc_oof':roc_auc_score(manifest.target,oof),'brier_oof':brier_score_loss(manifest.target,oof)}])
display(overall);overall.to_csv(OUTPUT_DIR/'C2_overall_metrics.csv',index=False)

## 10. Comparación directa con la CNN 3D anterior

In [ ]:
OLD_OOF=PROJECT_DIR/'cnn3d_local_2ch_v1'/'protocol_grouped_oof.csv'
if not OLD_OOF.exists():raise FileNotFoundError(OLD_OOF)
old=pd.read_csv(OLD_OOF).query('protocol_cluster == @TARGET_CLUSTER')[['uid','target','prediction']].rename(columns={'prediction':'prediction_3d'})
comparison=old.merge(oof_frame[['uid','prediction_25d']],on='uid',validate='one_to_one')
rows=[]
for name,col in [('CNN3D_leave_C2_out','prediction_3d'),('CNN25D_internal_C2','prediction_25d')]:
    rows.append({'model':name,'n':len(comparison),'logloss':log_loss(comparison.target,comparison[col]),
                 'auc':roc_auc_score(comparison.target,comparison[col]),'brier':brier_score_loss(comparison.target,comparison[col])})
comparison_metrics=pd.DataFrame(rows);display(comparison_metrics);comparison_metrics.to_csv(OUTPUT_DIR/'comparacion_C2_3D_vs_25D.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,(name,col) in zip(axes,[('CNN 3D sin C2','prediction_3d'),('CNN 2.5D interna C2','prediction_25d')]):
    sns.histplot(data=comparison,x=col,hue='target',bins=np.linspace(0,1,21),stat='density',common_norm=False,element='step',fill=False,ax=ax)
    ax.set_title(name);ax.set_xlabel('Probabilidad patológica')
plt.tight_layout();plt.savefig(OUTPUT_DIR/'comparacion_distribuciones_C2.png',dpi=170,bbox_inches='tight');plt.show()

## 11. Regla de decisión

In [ ]:
m25=overall.iloc[0];m3=comparison_metrics.iloc[0]
if m25.auc_oof>=.75 and m25.logloss_oof<=m3.logloss-.03:
    conclusion='La representación 2.5D recupera señal en C2. Siguiente paso: extender el modelo a todos los protocolos y repetir la CV agrupada.'
elif m25.auc_oof>=.68:
    conclusion='La representación 2.5D mejora parcialmente C2. Conviene probar ensamble 2.5D + CNN 3D antes de extender el entrenamiento completo.'
else:
    conclusion='La representación 2.5D no recupera suficiente señal. El problema de C2 excede la interpolación y requiere estudiar adquisición, etiquetas o adaptación de dominio.'
print(conclusion)
decision=pd.Series({'cnn25d_logloss':m25.logloss_oof,'cnn25d_auc':m25.auc_oof,'cnn25d_brier':m25.brier_oof,
                    'cnn3d_C2_logloss':m3.logloss,'cnn3d_C2_auc':m3.auc,'conclusion':conclusion},name='resultado').to_frame()
display(decision);decision.to_csv(OUTPUT_DIR/'decision_C2.csv')

## Criterio científico

La comparación no enfrenta dos esquemas de validación equivalentes: la CNN 3D fue evaluada en C2 sin haber visto ese protocolo, mientras la CNN 2.5D se entrena y valida dentro de C2. Por tanto:

- una mejora 2.5D demuestra que los cortes originales contienen señal aprovechable;
- no demuestra todavía generalización hacia protocolos desconocidos;
- si 2.5D funciona, la prueba siguiente será fijar la arquitectura y repetir leave-protocol-out sobre los seis clusters.